# Day 49: Hyperparameter Tuning

## Why Tune Hyperparameters?
Hyperparameters (like `n_estimators` or `max_depth` for a random forest) are settings you choose before training, as opposed to parameters the model learns from data. Scikit-learn's defaults are reasonable starting points, but they're not tailored to any specific dataset, so the best-performing combination almost always differs from the defaults - sometimes by a little, sometimes by a lot. Hyperparameter tuning is the systematic process of trying different combinations and measuring which one performs best, typically using cross-validation so the choice isn't overfit to a single train/test split. This notebook covers two standard approaches - exhaustive grid search and randomized search - and shows how to validate the final chosen model on a true held-out test set. The cell below just loads the Iris dataset and splits it, giving us data to tune against.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)  # test set stays untouched until final evaluation

## Grid Search
`GridSearchCV` performs an exhaustive search: it tries every possible combination of the values you list in `param_grid`, evaluating each one using k-fold cross-validation (splitting the training data into `cv` folds, training on some and validating on the rest, repeated so every fold is used for validation once) rather than a single split, which gives a more reliable performance estimate for each combination. With 2 values for `n_estimators` and 3 for `max_depth`, that's 2 x 3 = 6 combinations, each evaluated across 5 folds (`cv=5`), so 30 total model fits. This guarantees finding the best combination within the grid you defined, but the number of fits grows multiplicatively with the number of hyperparameters and values, so it becomes expensive fast on larger search spaces. `best_params_` shows which combination scored highest, and `best_score_` is that combination's average cross-validation accuracy (not test accuracy - that's checked separately later).

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {'n_estimators': [50, 100], 'max_depth': [3, 5, None]}  # None means trees grow until leaves are pure
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=5)  # 5-fold cross-validation per combination
grid.fit(X_train, y_train)  # trains a model for every combination x every fold
print('Best params:', grid.best_params_)
print('Best CV score:', grid.best_score_)  # average cross-validation accuracy of the best combination

## Randomized Search
When the search space gets large (many hyperparameters, many possible values each), trying every combination with grid search becomes computationally infeasible. `RandomizedSearchCV` instead samples a fixed number (`n_iter`) of random combinations from the specified distributions/lists and evaluates only those, still using cross-validation for each. This trades a guarantee of finding the absolute best combination in the grid for a large reduction in computation - in practice, random search often finds a combination nearly as good as full grid search in a fraction of the time, especially when only a few hyperparameters actually matter much. Here `param_dist` has 4 x 4 = 16 possible combinations, but we only sample `n_iter=5` of them rather than testing all 16, and `random_state=42` makes which 5 get picked reproducible.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {'n_estimators': [50, 100, 150, 200], 'max_depth': [3, 5, 7, None]}  # 16 possible combinations total
random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42), param_dist, n_iter=5, cv=5, random_state=42)  # only sample 5 of the 16 combinations
random_search.fit(X_train, y_train)
print('Best params:', random_search.best_params_)

## Evaluating the Tuned Model
Cross-validation scores from the search process (`best_score_`) are still computed only on the training data, so they can be slightly optimistic or just not reflect real-world performance as reliably as a genuinely untouched test set. The final, essential step of any tuning workflow is to take the best model the search found and evaluate it once on the held-out test set that played no role in either training or hyperparameter selection - this is the most trustworthy estimate of how the model will perform on new, unseen data. `grid.best_estimator_` is the actual fitted `RandomForestClassifier` with the best-found hyperparameters, already refit on the full training set, ready to score directly.

In [ ]:
best_model = grid.best_estimator_  # the refit model using the best hyperparameters found by grid search
print('Test accuracy:', best_model.score(X_test, y_test))  # true generalization check on data never used for training or tuning